# 4. Model Evaluation & Inference Pipeline 

In [1]:
import os, sys
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
for _c in (_HERE, os.path.dirname(_HERE)):
    _p = os.path.join(_c, "app")
    if os.path.isdir(_p):
        sys.path.insert(0, _p); break

import joblib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from text_utils import CLEAN_CSV, MODELS_DIR, probs_to_outputs, clean_review_text

## 2. Load Trained Model

In [2]:
bundle = joblib.load(os.path.join(MODELS_DIR, "review_model.pkl"))
pipe, classes = bundle["pipeline"], np.asarray(bundle["classes"])
df = pd.read_csv(CLEAN_CSV)
X, y = df["review_clean"].astype(str).values, df["Rating"].astype(int).values
print("loaded model trained with C =", bundle["best_C"])

loaded model trained with C = 3.0


In [3]:
X = [str(t) for t in df["review_clean"].tolist()]
y = df["Rating"].astype(int).to_numpy()
print(type(X), len(X))  

<class 'list'> 577


## 3. Cross-Validation Evaluation

In [4]:
cv_proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             method="predict_proba", n_jobs=1) 
ev = (cv_proba * classes.astype(float)).sum(axis=1)
mse = ((y - ev) ** 2).mean()
print("Rating RMSE : %.3f" % np.sqrt(mse))
print("Rating MAE  : %.3f" % mean_absolute_error(y, ev))
print("Rating R2   : %.3f" % r2_score(y, ev))
print("Predicted range: %.2f - %.2f  (old Ridge was stuck at 2.9-4.5)" % (ev.min(), ev.max()))

Rating RMSE : 0.855
Rating MAE  : 0.662
Rating R2   : 0.577
Predicted range: 1.03 - 4.94  (old Ridge was stuck at 2.9-4.5)
